# 대분류(6종) 카테고리 + 색상 고도화 검증 노트북

**스파이크/검증용 코드입니다. 프로덕션 코드가 아닙니다.**

`product_info_check.ipynb`(소분류 30종 + 카테고리 프롬프트 v1~v4 검증)와 별개 노트북이다 —
이번엔 **대분류(OUTER/TOP/BOTTOM/SHOES/BAG/ACCESSORY) 6종만** 분류했을 때 CLIP과 VLM이
각각 어떤지 비교하고, 색상은 지금까지 나온 개선안(프롬프트 v2/v3 + 화이트밸런스 전처리 +
팔레트 스와치 참조)을 한 자리에서 재실측한다.

## 사용법

1. §2(대분류)·§3(대분류 VLM)·§5(색상 VLM 계열)는 `RUN_VLM = True`로 바꿔야 실행된다
   (Qwen3-VL 로드 — 가볍지 않다).
2. 정답셋은 `product_info_check.ipynb`와 같은 `data/ground_truth.json`을 그대로 쓴다 —
   대분류 정답은 `taxonomy.major_of(subcategory)`로 즉석에서 계산한다(별도 라벨링 불필요).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import pandas as pd

from classify import classify_major_only, get_category_classifier
from color import PALETTE, build_swatch_reference
from dataset import load_ground_truth
from taxonomy import MAJOR_CATEGORIES, major_of

gt = load_ground_truth()
real_samples = [s for s in gt.samples if not s.synthetic]
print(f"정답셋: {len(gt.samples)}행(문제 {len(gt.issues)}행), 실제 사진 {len(real_samples)}장")
print(f"대분류 목록: {MAJOR_CATEGORIES}")


## 1. 설정 상수

`RUN_VLM`을 `True`로 바꿔야 아래 Qwen3-VL 관련 셀이 실행된다. `CATEGORY_TEMPLATE_KEY`는
CLIP 대분류 분류에 쓰는 템플릿(`taxonomy.MAJOR_PROMPT_TEMPLATES`의 키) — 소분류 검증에서
`en_clip_style`이 가장 좋았으므로 기본값으로 그대로 가져온다.

In [ ]:
RUN_VLM = False  # <- 명시적으로 True로 바꿔야 Qwen3-VL 관련 셀이 실행됨
CATEGORY_BACKEND = "clip_vit_b32"  # classify.CATEGORY_BACKENDS의 키
CATEGORY_TEMPLATE_KEY = "en_clip_style"  # taxonomy.MAJOR_PROMPT_TEMPLATES의 키

print(f"CATEGORY_BACKEND={CATEGORY_BACKEND}, CATEGORY_TEMPLATE_KEY={CATEGORY_TEMPLATE_KEY}, RUN_VLM={RUN_VLM}")


## 2. 대분류 분류 — CLIP zero-shot

`classify.classify_major_only()` — 대분류 6종만 물어본다(소분류 30-way보다 후보가 훨씬
적어 CLIP이 더 잘할 가능성이 있다).

In [ ]:
clip_classifier = get_category_classifier(CATEGORY_BACKEND)

clip_major_rows = []
for s in real_samples:
    result = classify_major_only(clip_classifier, s.image_path, CATEGORY_TEMPLATE_KEY)
    truth_major = major_of(s.subcategory)
    clip_major_rows.append(
        {
            "image_file": s.image_file,
            "truth_major": truth_major,
            "pred_major": result.predicted,
            "correct": result.predicted == truth_major,
            "score": round(result.score, 3),
        }
    )

clip_major_df = pd.DataFrame(clip_major_rows)
clip_major_acc = clip_major_df["correct"].mean()
print(f"[CLIP {CATEGORY_BACKEND}/{CATEGORY_TEMPLATE_KEY}] 대분류 top1 = {clip_major_acc:.2f}")
clip_major_df


## 3. 대분류 분류 — VLM(Qwen3-VL)

`vlm_control.query_major_category()` — 대분류 6종 프롬프트에 OUTER/TOP, BAG/ACCESSORY
구분 기준을 처음부터 넣어뒀다(소분류 v4의 교훈: 구조적/관찰 가능한 기준만 사용).

In [ ]:
if RUN_VLM:
    from vlm_control import query_major_category

    vlm_major_rows = []
    for s in real_samples:
        pred_major, raw = query_major_category(s.image_path)
        truth_major = major_of(s.subcategory)
        vlm_major_rows.append(
            {
                "image_file": s.image_file,
                "truth_major": truth_major,
                "pred_major": pred_major,
                "correct": pred_major == truth_major,
                "raw": raw,
            }
        )
    vlm_major_df = pd.DataFrame(vlm_major_rows)
    vlm_major_acc = vlm_major_df["correct"].mean()
    print(f"[VLM] 대분류 top1 = {vlm_major_acc:.2f}  (CLIP 대비 {clip_major_acc:.2f})")
    display(vlm_major_df)
else:
    print("RUN_VLM=False — 위 §1 셀에서 True로 바꾸고 재실행할 것.")


## 4. 대분류 비교 요약

CLIP(zero-shot, CPU 가능)과 VLM(Qwen, GPU 필요) 중 대분류 6-way만 놓고 보면 어느 쪽이
나은지, 그리고 소분류 30-way 결과(`product_info_check.ipynb` §3/§7-1)와 비교해 "분류
난이도를 낮추면 CLIP도 쓸 만해지는가"를 확인한다.

In [ ]:
summary_rows = [{"backend": f"CLIP({CATEGORY_BACKEND}/{CATEGORY_TEMPLATE_KEY})", "major_acc": clip_major_acc}]
if RUN_VLM:
    summary_rows.append({"backend": "VLM(qwen3-vl-4b-instruct)", "major_acc": vlm_major_acc})
pd.DataFrame(summary_rows)


## 5. 색상 분류 — 기존 설정 그대로 + 프롬프트/전처리 고도화 재실측

`product_info_check.ipynb`와 같은 정답셋·팔레트를 그대로 쓴다. 지금까지 나온 개선안을
전부 한 자리에서 비교한다:

- CLIP zero-shot(`clip_vit_b32`) — 기준선
- VLM v1(화이트밸런스 기본 적용, `query_color()` 기본값) — `REPORT.md` §13 실측 0.90
- VLM v2_anchored / v3_khaki_gray + 화이트밸런스 — **이 조합은 처음 테스트한다.**
  지난번 v2/v3 실측(§12)은 화이트밸런스 적용 **전**이었다 — 조명 편향 자체가 v2/v3
  회귀의 숨은 원인이었을 수 있어서, 화이트밸런스로 입력을 먼저 정리한 뒤에도 v2/v3가
  여전히 v1보다 나쁜지 다시 확인한다.

In [ ]:
from evaluate import evaluate_color

clip_color_preds = evaluate_color(gt.samples, "clip_vit_b32")
clip_color_acc = sum(p.correct_primary for p in clip_color_preds) / len(clip_color_preds)
print(f"[CLIP clip_vit_b32] 색상 primary_acc = {clip_color_acc:.2f}")


In [ ]:
if RUN_VLM:
    from vlm_control import COLOR_PROMPT_VERSIONS, query_color

    color_prompt_wb_rows = []
    for version in COLOR_PROMPT_VERSIONS:
        correct = 0
        for s in real_samples:
            pred_primary, _sec, _raw = query_color(s.image_path, prompt_version=version)
            correct += pred_primary == s.primary_color
        color_prompt_wb_rows.append(
            {
                "prompt_version": version,
                "white_balance": True,
                "n": len(real_samples),
                "primary_acc": correct / len(real_samples),
            }
        )
    color_prompt_wb_df = pd.DataFrame(color_prompt_wb_rows)
    display(color_prompt_wb_df)
else:
    print("RUN_VLM=False — §1 셀에서 True로 바꾸고 재실행할 것.")


## 6. 색상 분류 — 팔레트 스와치 참조(색상 개선안 4번)

`vlm_control.query_color_with_swatches()` — 상품 사진 아래에 팔레트 hex 스와치 그리드를
합성해 같은 이미지로 같이 보여준다(`color.build_swatch_reference()`). 텍스트로 색 이름을
"기억"해서 매칭하는 대신, 이미지 대 이미지로 비교 판단하게 유도하는 방법 — 여태 프롬프트
문구만 바꾸는 시도(v2/v3)는 다 whack-a-mole이었으니, 아예 다른 축(시각적 참조)을 시도해
본다. 화이트밸런스는 여기서도 기본 적용된다(상품 사진 쪽에만 적용, 스와치는 고정 hex라
보정 대상이 아니다).

In [ ]:
if RUN_VLM:
    from vlm_control import query_color_with_swatches

    swatch_rows = []
    for s in real_samples:
        pred_primary, pred_secondary, _raw = query_color_with_swatches(s.image_path)
        swatch_rows.append(
            {
                "image_file": s.image_file,
                "truth_primary": s.primary_color,
                "pred_primary": pred_primary,
                "pred_secondary": pred_secondary,
                "correct": pred_primary == s.primary_color,
            }
        )
    swatch_df = pd.DataFrame(swatch_rows)
    swatch_acc = swatch_df["correct"].mean()
    print(f"[VLM + 스와치 참조] 색상 primary_acc = {swatch_acc:.2f}")
    display(swatch_df)
else:
    print("RUN_VLM=False — §1 셀에서 True로 바꾸고 재실행할 것.")


## 7. 색상 비교 요약

지금까지 실측한 색상 방법론을 전부 한 표에 모은다 — `REPORT.md` §8~§13에 흩어져 있던
CLIP/VLM/프롬프트/전처리/스와치 결과의 최종 스코어보드.

In [ ]:
color_summary_rows = [{"method": "CLIP zero-shot(clip_vit_b32)", "primary_acc": clip_color_acc}]
if RUN_VLM:
    for _, row in color_prompt_wb_df.iterrows():
        color_summary_rows.append(
            {"method": f"VLM {row['prompt_version']} + 화이트밸런스", "primary_acc": row["primary_acc"]}
        )
    color_summary_rows.append({"method": "VLM v1 + 스와치 참조", "primary_acc": swatch_acc})
pd.DataFrame(color_summary_rows)


## 참고

- 소분류(30종) 결과·카테고리 프롬프트 v1~v4·색상 §8~§13 전체 서술은
  `product_info_check.ipynb`와 `REPORT.md` 참고.
- 이 노트북에서 나온 의미 있는 결과는 `REPORT.md`에도 반영해 둘 것 — 출력은 재실행하면
  사라진다.